In [1]:
# Persiapan SparkSession
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType
from pyspark.sql.functions import window, sum as spark_sum, count, col
import os
import shutil
import time
import pandas as pd
from datetime import datetime, timedelta

spark = SparkSession.builder \
    .appName("Tugas10-KopiNusantara") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")
print("SparkSession siap. Versi Spark:", spark.version)

26/11/07 20:05:29 WARN Utils: Your hostname, HP resolves to a loopback address: 127.0.1.1; using 192.168.0.105 instead (on interface wlo1)
26/11/07 20:05:29 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/07 20:05:30 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession siap. Versi Spark: 3.5.9


In [2]:
# A. Menyiapkan Streaming Source
skema_pesanan = StructType([
    StructField("pesanan_id", StringType(), True),
    StructField("nama_cabang", StringType(), True),
    StructField("nama_menu", StringType(), True),
    StructField("qty", IntegerType(), True),
    StructField("harga_satuan", IntegerType(), True),
    StructField("waktu_pesan", TimestampType(), True),
])

folder_tugas = "tugas10_stream"
shutil.rmtree(folder_tugas, ignore_errors=True)
os.makedirs(folder_tugas, exist_ok=True)

df_stream = spark.readStream \
    .format("csv") \
    .schema(skema_pesanan) \
    .option("header", True) \
    .load(folder_tugas)

print("Streaming DataFrame berhasil didefinisikan.")
print("df_stream.isStreaming:", df_stream.isStreaming)
df_stream.printSchema()

Streaming DataFrame berhasil didefinisikan.
df_stream.isStreaming: True
root
 |-- pesanan_id: string (nullable = true)
 |-- nama_cabang: string (nullable = true)
 |-- nama_menu: string (nullable = true)
 |-- qty: integer (nullable = true)
 |-- harga_satuan: integer (nullable = true)
 |-- waktu_pesan: timestamp (nullable = true)



In [3]:
# B. Agregasi per Cabang
df_stream_total = df_stream.withColumn(
    "total_harga",
    col("qty") * col("harga_satuan")
)

agregasi_cabang = df_stream_total.groupBy("nama_cabang").agg(
    spark_sum("total_harga").alias("total_pendapatan"),
    count("pesanan_id").alias("jumlah_pesanan")
)

query_cabang = agregasi_cabang.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("agregasi_cabang") \
    .start()

print("Streaming query agregasi cabang dimulai.")

Streaming query agregasi cabang dimulai.


In [4]:
# C. Agregasi Berbasis Window Waktu
df_stream_waktu = spark.readStream \
    .format("csv") \
    .schema(skema_pesanan) \
    .option("header", True) \
    .load(folder_tugas)

df_waktu_total = df_stream_waktu.withColumn(
    "total_harga",
    col("qty") * col("harga_satuan")
)

agregasi_waktu = df_waktu_total.groupBy(
    window(col("waktu_pesan"), "2 minutes")
).agg(
    spark_sum("total_harga").alias("total_pendapatan")
)

query_waktu = agregasi_waktu.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("agregasi_waktu") \
    .start()

print("Streaming query agregasi window waktu dimulai.")

Streaming query agregasi window waktu dimulai.


In [5]:
# D. Menguji dengan Data Simulasi
waktu_awal = datetime.now().replace(microsecond=0)

# Batch 1
pd.DataFrame({
    "pesanan_id": ["P1", "P2", "P3"],
    "nama_cabang": ["Magelang", "Yogyakarta", "Magelang"],
    "nama_menu": ["Kopi Tubruk", "Americano", "Cappuccino"],
    "qty": [2, 1, 3],
    "harga_satuan": [15000, 20000, 25000],
    "waktu_pesan": [
        waktu_awal.strftime("%Y-%m-%d %H:%M:%S"),
        (waktu_awal + timedelta(seconds=10)).strftime("%Y-%m-%d %H:%M:%S"),
        (waktu_awal + timedelta(seconds=20)).strftime("%Y-%m-%d %H:%M:%S")
    ],
}).to_csv(f"{folder_tugas}/batch_1.csv", index=False)

print("Batch 1 dikirim. Menunggu Spark memproses...")
time.sleep(8)
spark.sql("SELECT * FROM agregasi_cabang ORDER BY total_pendapatan DESC").show()
spark.sql("SELECT window.start, window.end, total_pendapatan FROM agregasi_waktu").show(truncate=False)

Batch 1 dikirim. Menunggu Spark memproses...


+-----------+----------------+--------------+
|nama_cabang|total_pendapatan|jumlah_pesanan|
+-----------+----------------+--------------+
|   Magelang|          105000|             2|
| Yogyakarta|           20000|             1|
+-----------+----------------+--------------+

+-------------------+-------------------+----------------+
|start              |end                |total_pendapatan|
+-------------------+-------------------+----------------+
|2026-11-07 20:30:00|2026-11-07 20:32:00|125000          |
+-------------------+-------------------+----------------+



In [6]:
# Batch 2
pd.DataFrame({
    "pesanan_id": ["P4", "P5", "P6"],
    "nama_cabang": ["Solo", "Magelang", "Yogyakarta"],
    "nama_menu": ["Espresso", "Kopi Tubruk", "Cafe Latte"],
    "qty": [2, 1, 2],
    "harga_satuan": [18000, 15000, 26000],
    "waktu_pesan": [
        (waktu_awal + timedelta(seconds=30)).strftime("%Y-%m-%d %H:%M:%S"),
        (waktu_awal + timedelta(seconds=40)).strftime("%Y-%m-%d %H:%M:%S"),
        (waktu_awal + timedelta(seconds=50)).strftime("%Y-%m-%d %H:%M:%S")
    ],
}).to_csv(f"{folder_tugas}/batch_2.csv", index=False)

print("Batch 2 dikirim. Menunggu Spark memproses...")
time.sleep(8)
spark.sql("SELECT * FROM agregasi_cabang ORDER BY total_pendapatan DESC").show()
spark.sql("SELECT window.start, window.end, total_pendapatan FROM agregasi_waktu").show(truncate=False)

Batch 2 dikirim. Menunggu Spark memproses...
+-----------+----------------+--------------+
|nama_cabang|total_pendapatan|jumlah_pesanan|
+-----------+----------------+--------------+
|   Magelang|          120000|             3|
| Yogyakarta|           72000|             2|
|       Solo|           36000|             1|
+-----------+----------------+--------------+

+-------------------+-------------------+----------------+
|start              |end                |total_pendapatan|
+-------------------+-------------------+----------------+
|2026-11-07 20:32:00|2026-11-07 20:34:00|67000           |
|2026-11-07 20:30:00|2026-11-07 20:32:00|161000          |
+-------------------+-------------------+----------------+



In [7]:
# Batch 3
pd.DataFrame({
    "pesanan_id": ["P7", "P8", "P9"],
    "nama_cabang": ["Yogyakarta", "Solo", "Magelang"],
    "nama_menu": ["Nasi Goreng", "Mendoan", "Mocha Latte"],
    "qty": [1, 2, 2],
    "harga_satuan": [28000, 10000, 30000],
    "waktu_pesan": [
        (waktu_awal + timedelta(seconds=60)).strftime("%Y-%m-%d %H:%M:%S"),
        (waktu_awal + timedelta(seconds=70)).strftime("%Y-%m-%d %H:%M:%S"),
        (waktu_awal + timedelta(seconds=80)).strftime("%Y-%m-%d %H:%M:%S")
    ],
}).to_csv(f"{folder_tugas}/batch_3.csv", index=False)

print("Batch 3 dikirim. Menunggu Spark memproses...")
time.sleep(8)
spark.sql("SELECT * FROM agregasi_cabang ORDER BY total_pendapatan DESC").show()
spark.sql("SELECT window.start, window.end, total_pendapatan FROM agregasi_waktu").show(truncate=False)

Batch 3 dikirim. Menunggu Spark memproses...
+-----------+----------------+--------------+
|nama_cabang|total_pendapatan|jumlah_pesanan|
+-----------+----------------+--------------+
|   Magelang|          180000|             4|
| Yogyakarta|          100000|             3|
|       Solo|           56000|             2|
+-----------+----------------+--------------+

+-------------------+-------------------+----------------+
|start              |end                |total_pendapatan|
+-------------------+-------------------+----------------+
|2026-11-07 20:32:00|2026-11-07 20:34:00|175000          |
|2026-11-07 20:30:00|2026-11-07 20:32:00|161000          |
+-------------------+-------------------+----------------+



In [8]:
# E. Sink Berbasis Berkas (Tantangan
shutil.rmtree("tugas10_output", ignore_errors=True)
shutil.rmtree("tugas10_checkpoint", ignore_errors=True)

query_sink = df_stream.writeStream \
    .format("parquet") \
    .option("path", "tugas10_output") \
    .option("checkpointLocation", "tugas10_checkpoint") \
    .outputMode("append") \
    .start()

print("Streaming query Parquet dimulai.")

Streaming query Parquet dimulai.


In [9]:
waktu_batch4 = datetime.now().replace(microsecond=0)
pd.DataFrame({
    "pesanan_id": ["P10", "P11"],
    "nama_cabang": ["Magelang", "Solo"],
    "nama_menu": ["Pisang Goreng", "Mie Goreng"],
    "qty": [1, 2],
    "harga_satuan": [15000, 24000],
    "waktu_pesan": [
        waktu_batch4.strftime("%Y-%m-%d %H:%M:%S"),
        (waktu_batch4 + timedelta(seconds=5)).strftime("%Y-%m-%d %H:%M:%S")
    ],
}).to_csv(f"{folder_tugas}/batch_4.csv", index=False)

print("Batch 4 dikirim. Menunggu Spark menulis Parquet...")
time.sleep(10)
spark.sql("SELECT * FROM agregasi_cabang ORDER BY total_pendapatan DESC").show()
spark.sql("SELECT window.start, window.end, total_pendapatan FROM agregasi_waktu").show(truncate=False)

# Memeriksa isi folder output
!ls tugas10_output

Batch 4 dikirim. Menunggu Spark menulis Parquet...
+-----------+----------------+--------------+
|nama_cabang|total_pendapatan|jumlah_pesanan|
+-----------+----------------+--------------+
|   Magelang|          195000|             5|
|       Solo|          104000|             3|
| Yogyakarta|          100000|             3|
+-----------+----------------+--------------+

+-------------------+-------------------+----------------+
|start              |end                |total_pendapatan|
+-------------------+-------------------+----------------+
|2026-11-07 20:32:00|2026-11-07 20:34:00|175000          |
|2026-11-07 20:30:00|2026-11-07 20:32:00|161000          |
|2026-11-07 20:42:00|2026-11-07 20:44:00|63000           |
+-------------------+-------------------+----------------+

part-00000-31e7661e-45c4-49c9-b31d-dc34660e0487-c000.snappy.parquet
part-00000-611d2379-2244-44d4-8859-9c5647b6aa05-c000.snappy.parquet
part-00001-8e4b4111-5bfb-4024-a212-78b3b562acaa-c000.snappy.parquet
part-000

In [10]:
df_parquet = spark.read.parquet("tugas10_output")
print("Data Parquet berhasil dibaca kembali:")
df_parquet.show(truncate=False)
print("Jumlah baris:", df_parquet.count())

Data Parquet berhasil dibaca kembali:
+----------+-----------+-------------+---+------------+-------------------+
|pesanan_id|nama_cabang|nama_menu    |qty|harga_satuan|waktu_pesan        |
+----------+-----------+-------------+---+------------+-------------------+
|P4        |Solo       |Espresso     |2  |18000       |2026-11-07 20:31:59|
|P5        |Magelang   |Kopi Tubruk  |1  |15000       |2026-11-07 20:32:09|
|P6        |Yogyakarta |Cafe Latte   |2  |26000       |2026-11-07 20:32:19|
|P1        |Magelang   |Kopi Tubruk  |2  |15000       |2026-11-07 20:31:29|
|P2        |Yogyakarta |Americano    |1  |20000       |2026-11-07 20:31:39|
|P3        |Magelang   |Cappuccino   |3  |25000       |2026-11-07 20:31:49|
|P7        |Yogyakarta |Nasi Goreng  |1  |28000       |2026-11-07 20:32:29|
|P8        |Solo       |Mendoan      |2  |10000       |2026-11-07 20:32:39|
|P9        |Magelang   |Mocha Latte  |2  |30000       |2026-11-07 20:32:49|
|P10       |Magelang   |Pisang Goreng|1  |15000   

In [11]:
query_cabang.stop()
query_waktu.stop()
query_sink.stop()
print("Seluruh streaming query dihentikan.")

spark.stop()
print("SparkSession ditutup.")

Seluruh streaming query dihentikan.
SparkSession ditutup.
